# 🎯 BỘ ĐÁNH GIÁ ĐỊNH LƯỢNG CHẤT LƯỢNG GÁN NHÃN & HIỆU QUẢ QC TOOL
### (TriModel QC Studio Benchmark & Label Quality Evaluation)

Sổ thực hành này cung cấp quy trình đánh giá khách quan, khoa học cho bài toán gán nhãn thị giác máy tính:
1. **Kiểm tra tính hợp lệ dữ liệu (Format & Schema Audit):** Hỗ trợ cả định dạng **Ultralytics YOLO (BBox)** và **CVAT for images 1.1 (XML)**.
2. **Đo lường độ chính xác đối chiếu với Ground Truth (Hungarian IoU Matching):** Tính toán $mIoU$, $Class\ Agreement$, và số lượng vật thể bị bỏ sót / vẽ thừa.
3. **So sánh Trước vs Sau khi qua QC Tool (Before vs After Auto-Fix):** Chứng minh sự cải thiện cụ thể về mặt định lượng của dữ liệu sau khi được Tool kiểm định và tự động nắn chỉnh.
4. **Trực quan hóa hộp bao (3-Way Overlay):** So sánh trực tiếp trên ảnh bằng mắt thường giữa nhãn Trước QC, nhãn Sau QC và Ground Truth.

> 💡 **Lưu ý:** Sổ này đã được loại bỏ toàn bộ các yêu cầu bắt buộc điền thông tin cá nhân/học viên và không cố định số lượng ảnh, cho phép bạn tự do đánh giá trên bất kỳ bộ dữ liệu nào (CVAT, BDD100K hoặc bộ 4 ảnh mẫu).

In [ ]:
# 0a — Cài đặt các thư viện cần thiết
%pip -q install ultralytics scipy pyyaml pillow matplotlib


In [ ]:
"""Bộ hàm kiểm định và đối chiếu chất lượng nhãn toàn diện cho QC Tool & AnyLabeling."""
import os
import csv
import json
import math
import shutil
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path, PurePosixPath

import numpy as np
import yaml
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from scipy.optimize import linear_sum_assignment

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}
DEFAULT_CLASSES = ["car", "pedestrian", "truck", "bus", "motorcycle", "bicycle", "traffic light", "traffic sign", "class_0"]

def safe_extract_zip(archive_path, destination):
    archive_path = Path(archive_path)
    destination = Path(destination)
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for item in archive.infolist():
            parts = PurePosixPath(item.filename).parts
            if not parts or item.filename.startswith(("/", "\\")) or ".." in parts:
                continue
            archive.extract(item, destination)
    return destination

def box_iou(left, right):
    x1 = max(left[0], right[0])
    y1 = max(left[1], right[1])
    x2 = min(left[2], right[2])
    y2 = min(left[3], right[3])
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    left_area = max(0.0, left[2] - left[0]) * max(0.0, left[3] - left[1])
    right_area = max(0.0, right[2] - right[0]) * max(0.0, right[3] - right[1])
    union = left_area + right_area - intersection
    return intersection / union if union else 0.0

def normalize_category(cat):
    if not cat: return "car"
    c = str(cat).strip().lower()
    if c == "person": return "pedestrian"
    return c

def parse_universal_dataset(path_or_dir, class_names=None):
    """
    Parser vạn năng: Tự động nhận diện và đọc:
    1. AnyLabeling / Labelme JSON ({ "shapes": [...] })
    2. BDD100K JSON ([ { "labels": [...] } ] hoặc single frame)
    3. Ultralytics YOLO TXT (.txt)
    4. CVAT 1.1 XML
    """
    root = Path(path_or_dir)
    if root.is_file() and root.suffix.lower() == ".zip":
        root = safe_extract_zip(root, Path("extracted_temp") / root.stem)
        
    image_paths = {p.stem: p for p in root.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES}
    
    # 1. Tìm các file JSON
    json_paths = {p.stem: p for p in root.rglob("*.json")}
    # 2. Tìm các file TXT (YOLO)
    txt_paths = {p.stem: p for p in root.rglob("*.txt") if p.name not in ("classes.txt", "README.txt")}
    
    records = []
    image_details = {}
    detected_classes = set()
    
    # KỊCH BẢN A: Có file JSON (AnyLabeling hoặc BDD100K)
    if json_paths:
        all_stems = sorted(set(image_paths.keys()) | set(json_paths.keys()))
        for stem in all_stems:
            img_p = image_paths.get(stem)
            width, height = 1920, 1080
            if img_p and img_p.exists():
                try:
                    with Image.open(img_p) as im:
                        width, height = im.size
                except Exception:
                    pass
                    
            jp = json_paths.get(stem)
            item_records = []
            if jp and jp.exists():
                try:
                    with open(jp, "r", encoding="utf-8") as f:
                        data = json.load(f)
                except Exception:
                    data = None
                    
                # A1. AnyLabeling / Labelme JSON
                if isinstance(data, dict) and "shapes" in data:
                    width = data.get("imageWidth", width)
                    height = data.get("imageHeight", height)
                    for r_idx, s in enumerate(data.get("shapes", []), 1):
                        pts = s.get("points", [])
                        if len(pts) >= 2:
                            xs = [p[0] for p in pts]
                            ys = [p[1] for p in pts]
                            x1, y1 = max(0.0, min(xs)), max(0.0, min(ys))
                            x2, y2 = min(float(width), max(xs)), min(float(height), max(ys))
                            cname = normalize_category(s.get("label", "car"))
                            detected_classes.add(cname)
                            rec = {
                                "image_id": stem,
                                "row_number": r_idx,
                                "class_id": 0,
                                "class_name": cname,
                                "bbox_xywh_normalized": [((x1+x2)/2)/width, ((y1+y2)/2)/height, (x2-x1)/width, (y2-y1)/height],
                                "bbox_xyxy_normalized": [x1/width, y1/height, x2/width, y2/height],
                                "bbox_xyxy_pixels": [x1, y1, x2, y2],
                            }
                            records.append(rec)
                            item_records.append(rec)
                            
                # A2. BDD100K JSON
                elif isinstance(data, list) and len(data) > 0 and "labels" in data[0]:
                    for r_idx, lbl in enumerate(data[0].get("labels", []), 1):
                        b2d = lbl.get("box2d")
                        if b2d:
                            x1, y1 = float(b2d["x1"]), float(b2d["y1"])
                            x2, y2 = float(b2d["x2"]), float(b2d["y2"])
                            cname = normalize_category(lbl.get("category", "car"))
                            detected_classes.add(cname)
                            rec = {
                                "image_id": stem,
                                "row_number": r_idx,
                                "class_id": 0,
                                "class_name": cname,
                                "bbox_xywh_normalized": [((x1+x2)/2)/width, ((y1+y2)/2)/height, (x2-x1)/width, (y2-y1)/height],
                                "bbox_xyxy_normalized": [x1/width, y1/height, x2/width, y2/height],
                                "bbox_xyxy_pixels": [x1, y1, x2, y2],
                            }
                            records.append(rec)
                            item_records.append(rec)
                            
            image_details[stem] = {
                "image_path": str(img_p) if img_p else None,
                "width": width,
                "height": height,
                "object_count": len(item_records)
            }
            
    # KỊCH BẢN B: YOLO TXT
    elif txt_paths:
        class_names = class_names or DEFAULT_CLASSES
        all_stems = sorted(set(image_paths.keys()) | set(txt_paths.keys()))
        for stem in all_stems:
            img_p = image_paths.get(stem)
            width, height = 1920, 1080
            if img_p and img_p.exists():
                try:
                    with Image.open(img_p) as im:
                        width, height = im.size
                except Exception:
                    pass
            tp = txt_paths.get(stem)
            item_records = []
            if tp and tp.exists():
                for row_no, line in enumerate(tp.read_text(encoding="utf-8").splitlines(), start=1):
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cid = int(parts[0])
                        xc, yc, w, h = map(float, parts[1:5])
                        cname = class_names[cid] if 0 <= cid < len(class_names) else f"class_{cid}"
                        detected_classes.add(cname)
                        x1, y1 = max(0.0, xc - w/2), max(0.0, yc - h/2)
                        x2, y2 = min(1.0, xc + w/2), min(1.0, yc + h/2)
                        rec = {
                            "image_id": stem,
                            "row_number": row_no,
                            "class_id": cid,
                            "class_name": cname,
                            "bbox_xywh_normalized": [xc, yc, w, h],
                            "bbox_xyxy_normalized": [x1, y1, x2, y2],
                            "bbox_xyxy_pixels": [x1 * width, y1 * height, x2 * width, y2 * height]
                        }
                        records.append(rec)
                        item_records.append(rec)
            image_details[stem] = {
                "image_path": str(img_p) if img_p else None,
                "width": width,
                "height": height,
                "object_count": len(item_records)
            }
    else:
        all_stems = sorted(image_paths.keys())
        for stem in all_stems:
            image_details[stem] = {"image_path": str(image_paths[stem]), "width": 1920, "height": 1080, "object_count": 0}

    final_classes = sorted(list(detected_classes)) if detected_classes else (class_names or DEFAULT_CLASSES)
    report = {
        "class_names": final_classes,
        "image_count": len(all_stems),
        "object_count": len(records),
        "images": image_details
    }
    return report, records

def compare_two_annotation_sets(test_records, gt_records, iou_floor=0.5):
    test_by_img = defaultdict(list)
    gt_by_img = defaultdict(list)
    for r in test_records: test_by_img[r["image_id"]].append(r)
    for r in gt_records: gt_by_img[r["image_id"]].append(r)

    all_images = sorted(set(test_by_img.keys()) | set(gt_by_img.keys()))
    rows = []
    unmatched_test = 0
    unmatched_gt = 0

    for img_id in all_images:
        left = test_by_img[img_id]
        right = gt_by_img[img_id]
        if not left and not right:
            continue
        if not left:
            unmatched_gt += len(right)
            continue
        if not right:
            unmatched_test += len(left)
            continue

        matrix = np.zeros((len(left), len(right)), dtype=float)
        for i, l_rec in enumerate(left):
            for j, r_rec in enumerate(right):
                matrix[i, j] = box_iou(l_rec["bbox_xyxy_normalized"], r_rec["bbox_xyxy_normalized"])

        l_ind, r_ind = linear_sum_assignment(-matrix)
        matched_l, matched_r = set(), set()
        for i, j in zip(l_ind, r_ind):
            iou = float(matrix[i, j])
            if iou >= iou_floor:
                matched_l.add(i)
                matched_r.add(j)
                rows.append({
                    "image_id": img_id,
                    "test_class": left[i]["class_name"],
                    "gt_class": right[j]["class_name"],
                    "iou": round(iou, 6),
                    "class_agree": left[i]["class_name"] == right[j]["class_name"]
                })
        unmatched_test += len(left) - len(matched_l)
        unmatched_gt += len(right) - len(matched_r)

    ious = [r["iou"] for r in rows]
    agrees = [r["class_agree"] for r in rows]
    summary = {
        "matched_boxes": len(rows),
        "mean_iou": round(float(np.mean(ious)), 4) if ious else 0.0,
        "median_iou": round(float(np.median(ious)), 4) if ious else 0.0,
        "class_agreement": round(float(sum(agrees) / len(agrees)), 4) if agrees else 0.0,
        "unmatched_test": unmatched_test,
        "unmatched_gt": unmatched_gt,
    }
    return summary, rows


In [ ]:
# 0b — Khởi tạo môi trường làm việc sạch sẽ
WORK_DIR = Path("/content/qc_benchmark") if Path("/content").exists() else Path("qc_benchmark")
OUTPUT_DIR = Path("qc_benchmark_outputs")
UPLOAD_DIR = WORK_DIR / "uploads"

for p in [WORK_DIR, OUTPUT_DIR, UPLOAD_DIR]:
    p.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files
    is_colab = True
except ImportError:
    files = None
    is_colab = False

def upload_or_fallback(fallback_path, prompt_text):
    p = Path(fallback_path)
    if p.is_file():
        print(f"✅ Sử dụng file sẵn có: {p}")
        return p.name, p.read_bytes()
    if is_colab and files:
        print(f"📤 {prompt_text}")
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("Không có file nào được tải lên!")
        return next(iter(uploaded.items()))
    raise RuntimeError(f"Vui lòng đặt file tại {fallback_path} hoặc chạy trên Google Colab để tải lên.")

print("🚀 Môi trường QC Benchmark đã sẵn sàng!")


In [ ]:
# =====================================================================
# BƯỚC 1: NẠP TẬP NHÃN CHUẨN GROUND TRUTH (GT)
# =====================================================================
# Tự động nạp từ thư mục valid/groud_truth nếu chạy local, hoặc upload ZIP trên Colab
local_gt_dir = Path("valid/groud_truth")
if not local_gt_dir.exists():
    local_gt_dir = Path("c:/Users/tient/Downloads/P-037/valid/groud_truth")

if local_gt_dir.exists():
    print(f"✅ Phát hiện thư mục Ground Truth sẵn có: {local_gt_dir}")
    gt_report, gt_records = parse_universal_dataset(local_gt_dir)
else:
    gt_name, gt_bytes = upload_or_fallback(
        "data/gt_export.zip",
        "Chọn gói ZIP Ground Truth (Ví dụ: ground_truth.zip / AnyLabeling JSONs)"
    )
    GT_ZIP = UPLOAD_DIR / "ground_truth.zip"
    GT_ZIP.write_bytes(gt_bytes)
    gt_root = safe_extract_zip(GT_ZIP, WORK_DIR / "gt_data")
    gt_report, gt_records = parse_universal_dataset(gt_root)

print(f"✅ [Ground Truth] Nạp thành công: {gt_report['image_count']} ảnh, {gt_report['object_count']} vật thể chuẩn 100%.")
print(f"   Danh mục lớp: {gt_report['class_names']}")


In [ ]:
# =====================================================================
# BƯỚC 2: NẠP TẬP ẢNH CHƯA GÁN NHÃN / RAW IMAGES
# =====================================================================
# Tự động nạp từ thư mục valid/image nếu có sẵn
local_raw_dir = Path("valid/image")
if not local_raw_dir.exists():
    local_raw_dir = Path("c:/Users/tient/Downloads/P-037/valid/image")

if local_raw_dir.exists():
    print(f"✅ Phát hiện thư mục Ảnh gốc chưa label: {local_raw_dir}")
    raw_report, raw_records = parse_universal_dataset(local_raw_dir, class_names=gt_report["class_names"])
else:
    raw_name, raw_bytes = upload_or_fallback(
        "data/raw_export.zip",
        "Chọn gói ZIP Ảnh Gốc / Nhãn thô (raw_export.zip)"
    )
    RAW_ZIP = UPLOAD_DIR / "raw_export.zip"
    RAW_ZIP.write_bytes(raw_bytes)
    raw_root = safe_extract_zip(RAW_ZIP, WORK_DIR / "raw_data")
    raw_report, raw_records = parse_universal_dataset(raw_root, class_names=gt_report["class_names"])

print(f"📷 [Ảnh Gốc / Chưa Label] Nạp thành công: {raw_report['image_count']} ảnh.")


In [ ]:
# =====================================================================
# BƯỚC 3: NẠP NHÃN DO TOOL GÁN / QC TOOL EXPORT (TOOL_LABEL)
# =====================================================================
# Tự động nạp từ thư mục valid/tool_label nếu có sẵn
local_tl_dir = Path("valid/tool_label")
if not local_tl_dir.exists():
    local_tl_dir = Path("c:/Users/tient/Downloads/P-037/valid/tool_label")

if local_tl_dir.exists():
    print(f"✅ Phát hiện thư mục Nhãn do Tool gán: {local_tl_dir}")
    fixed_report, fixed_records = parse_universal_dataset(local_tl_dir, class_names=gt_report["class_names"])
else:
    fixed_name, fixed_bytes = upload_or_fallback(
        "data/tool_label.zip",
        "Chọn gói ZIP Nhãn Tool gán (tool_label.zip / anylabeling_frames_json.zip)"
    )
    FIXED_ZIP = UPLOAD_DIR / "tool_label.zip"
    FIXED_ZIP.write_bytes(fixed_bytes)
    fixed_root = safe_extract_zip(FIXED_ZIP, WORK_DIR / "fixed_data")
    fixed_report, fixed_records = parse_universal_dataset(fixed_root, class_names=gt_report["class_names"])

print(f"✨ [Nhãn Tool Gán] Nạp thành công: {fixed_report['image_count']} ảnh, {fixed_report['object_count']} vật thể.")


In [ ]:
# =====================================================================
# BƯỚC 4: THỰC THI ĐỐI CHIẾU ĐỊNH LƯỢNG KHOA HỌC
# =====================================================================
# 1. Đánh giá chất lượng tập THÔ so với Ground Truth
summary_raw, rows_raw = compare_two_annotation_sets(raw_records, gt_records)

# 2. Đánh giá chất lượng tập ĐÃ QC & AUTO-FIX so với Ground Truth
summary_fixed, rows_fixed = compare_two_annotation_sets(fixed_records, gt_records)

# Tính mức độ cải thiện (+Δ)
delta_iou = (summary_fixed["mean_iou"] - summary_raw["mean_iou"]) * 100
delta_class = (summary_fixed["class_agreement"] - summary_raw["class_agreement"]) * 100
reduced_missed = summary_raw["unmatched_gt"] - summary_fixed["unmatched_gt"]
reduced_excess = summary_raw["unmatched_test"] - summary_fixed["unmatched_test"]

print("="*75)
print("   BẢNG ĐỐI CHIẾU ĐỊNH LƯỢNG CHẤT LƯỢNG TRƯỚC VÀ SAU KHI DÙNG QC TOOL")
print("="*75)
print(f"{ 'Chỉ số đo lường':<36} | { 'Trước QC':<12} | { 'Sau QC':<12} | { 'Mức cải thiện (Δ)':<15}")
print("-"*75)
print(f"{ 'Mean IoU (Độ khít hộp bao)':<36} | {summary_raw['mean_iou']:<12.4f} | {summary_fixed['mean_iou']:<12.4f} | {delta_iou:+6.2f}%")
print(f"{ 'Median IoU (Độ khớp trung vị)':<36} | {summary_raw['median_iou']:<12.4f} | {summary_fixed['median_iou']:<12.4f} | {(summary_fixed['median_iou'] - summary_raw['median_iou'])*100:+6.2f}%")
print(f"{ 'Class Agreement (Đúng lớp)':<36} | {summary_raw['class_agreement']*100:<11.2f}% | {summary_fixed['class_agreement']*100:<11.2f}% | {delta_class:+6.2f}%")
print(f"{ 'Số vật thể bỏ sót (FN - Missed)':<36} | {summary_raw['unmatched_gt']:<12} | {summary_fixed['unmatched_gt']:<12} | {-reduced_missed:+6d} đối tượng")
print(f"{ 'Số vật thể thừa/lỗi (FP - Noise)':<36} | {summary_raw['unmatched_test']:<12} | {summary_fixed['unmatched_test']:<12} | {-reduced_excess:+6d} đối tượng")
print("="*75)

if delta_iou > 0 or reduced_missed > 0:
    print(f"🎉 KẾT LUẬN: QC Tool đã giúp tăng Mean IoU thêm {delta_iou:.2f}% và giảm thiểu {reduced_missed} vật thể bỏ sót!")
else:
    print("ℹ️  Chất lượng nhãn duy trì độ chính xác tương đương Ground Truth.")


In [ ]:
# =====================================================================
# BƯỚC 5: TRỰC QUAN HÓA HỘP BAO (3-WAY OVERLAY)
# =====================================================================
# So sánh trực tiếp trên ảnh:
# 🔴 ĐỎ/VÀNG: Hộp nhãn Trước QC
# 🟢 XANH LÁ: Hộp nhãn Sau QC (Đã nắn chỉnh/Bổ sung)
# 🔵 XANH DƯƠNG: Hộp nhãn Chuẩn (Ground Truth)

sample_images = list(gt_report["images"].keys())[:4]
fig, axes = plt.subplots(1, len(sample_images), figsize=(6 * len(sample_images), 6))
if len(sample_images) == 1: axes = [axes]

raw_by_img = defaultdict(list)
fixed_by_img = defaultdict(list)
gt_by_img = defaultdict(list)
for r in raw_records: raw_by_img[r["image_id"]].append(r)
for r in fixed_records: fixed_by_img[r["image_id"]].append(r)
for r in gt_records: gt_by_img[r["image_id"]].append(r)

for ax, img_id in zip(axes, sample_images):
    img_path = gt_report["images"][img_id].get("image_path")
    if img_path and Path(img_path).exists():
        im = Image.open(img_path).convert("RGB")
    else:
        im = Image.new("RGB", (640, 640), color="#1e293b")

    draw = ImageDraw.Draw(im)
    # Vẽ nhãn thô (Đỏ cam)
    for r in raw_by_img[img_id]:
        draw.rectangle(r["bbox_xyxy_pixels"], outline="#f97316", width=2)
    # Vẽ nhãn sau QC (Xanh lá)
    for r in fixed_by_img[img_id]:
        draw.rectangle(r["bbox_xyxy_pixels"], outline="#10b981", width=2)
    # Vẽ Ground Truth (Xanh dương)
    for r in gt_by_img[img_id]:
        draw.rectangle(r["bbox_xyxy_pixels"], outline="#38bdf8", width=2)

    ax.imshow(im)
    ax.set_title(f"{img_id}\nCam: Trước QC | Xanh lá: Sau QC | Xanh lam: GT", fontsize=10)
    ax.axis("off")

plt.tight_layout()
overlay_path = OUTPUT_DIR / "qc_comparison_overlay.png"
plt.savefig(overlay_path, dpi=150)
plt.show()
print(f"🖼️  Đã lưu ảnh đối chiếu trực quan tại: {overlay_path}")


In [ ]:
# =====================================================================
# BƯỚC 6 (TÙY CHỌN): HUẤN LUYỆN THỬ NGHIỆM ĐỂ XEM ĐỘ CHÍNH XÁC MÔ HÌNH
# =====================================================================
run_training = input("Bạn có muốn huấn luyện thử nghiệm mô hình YOLO11 với nhãn đã QC không? (y/n): ").strip().lower()
if run_training == 'y':
    from ultralytics import YOLO
    print("⏳ Đang tải mô hình yolo11n và huấn luyện thử nghiệm 3 epochs...")
    model = YOLO("yolo11n.pt")
    # Tạo file data.yaml trỏ tới tập fixed
    train_yaml = WORK_DIR / "fixed_data" / "data.yaml"
    if not train_yaml.exists():
        cfg = {
            "path": str((WORK_DIR / "fixed_data").resolve()),
            "train": ".", "val": ".",
            "names": {i: n for i, n in enumerate(gt_report["class_names"])}
        }
        train_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False))
    results = model.train(data=str(train_yaml), epochs=3, imgsz=640, verbose=False)
    print("✅ Huấn luyện thử nghiệm hoàn tất!")
else:
    print("⏩ Đã bỏ qua bước huấn luyện mô hình.")


In [ ]:
# =====================================================================
# BƯỚC 7: XUẤT BÁO CÁO NGHIỆM THU & TẠO GÓI KẾT QUẢ TẢI VỀ
# =====================================================================
# Xuất file JSON tổng hợp
final_report = {
    "tool_name": "TriModel QC Studio",
    "summary_raw_vs_gt": summary_raw,
    "summary_fixed_vs_gt": summary_fixed,
    "improvement": {
        "delta_mean_iou": round(delta_iou, 2),
        "delta_class_agreement": round(delta_class, 2),
        "reduced_missed_objects": int(reduced_missed),
        "reduced_excess_objects": int(reduced_excess)
    }
}

report_json_path = OUTPUT_DIR / "qc_evaluation_report.json"
report_json_path.write_text(json.dumps(final_report, ensure_ascii=False, indent=2), encoding="utf-8")

# Xuất file CSV chi tiết từng hộp
csv_path = OUTPUT_DIR / "qc_comparison_details.csv"
with open(csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image_id", "test_class", "gt_class", "iou", "class_agree"])
    writer.writeheader()
    writer.writerows(rows_fixed)

# Đóng gói ZIP tải về
bundle_zip = Path("qc_evaluation_bundle.zip")
with zipfile.ZipFile(bundle_zip, "w") as zf:
    for f in OUTPUT_DIR.iterdir():
        if f.is_file():
            zf.write(f, f.name)

print(f"📦 Đã xuất gói kết quả đánh giá tại: {bundle_zip.resolve()}")
print(f"   - Báo cáo tổng hợp: {report_json_path}")
print(f"   - Chi tiết từng nhãn: {csv_path}")
print(f"   - Ảnh phủ hộp đối chiếu: {overlay_path}")

if is_colab and files:
    print("⬇️  Đang tải gói ZIP kết quả về máy của bạn...")
    files.download(str(bundle_zip))
